# 02 — Transformação + Detecção de Anomalias

Este notebook realiza a etapa **T (Transform)** do ETL:
- Padronização de tipos e datas
- Limpeza de valores nulos e duplicatas
- Consolidação das 3 fontes em um dataset unificado
- Detecção automática de anomalias via **Z-score (limiar 3σ)**

Dados processados são salvos em `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import os

os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Bibliotecas carregadas.")

## 2.1 — Carregar dados brutos extraídos

In [ ]:
df_fin = pd.read_parquet("data/raw/finance_csv.parquet")
df_ibov = pd.read_parquet("data/raw/ibovespa.parquet")
df_orc = pd.read_parquet("data/raw/orcamento.parquet")

print(f"Transações financeiras: {len(df_fin)} registros")
print(f"Ibovespa:               {len(df_ibov)} registros")
print(f"Orçamento:              {len(df_orc)} registros")

## 2.2 — Transformação do dataset financeiro

In [ ]:
# Antes: verificar tipos originais
print("Tipos ANTES da transformação:")
print(df_fin.dtypes)
print(f"\nNulos por coluna:\n{df_fin.isnull().sum()}")
print(f"\nDuplicatas: {df_fin.duplicated().sum()}")

In [ ]:
# Padronizar tipos
df_fin["date"] = pd.to_datetime(df_fin["date"])
df_fin["amount"] = df_fin["amount"].astype("float64")

# Remover duplicatas
antes = len(df_fin)
df_fin = df_fin.drop_duplicates()
print(f"Duplicatas removidas: {antes - len(df_fin)}")

# Preencher nulos (se houver)
df_fin = df_fin.dropna(subset=["date", "amount"])

# Ordenar por data
df_fin = df_fin.sort_values("date").reset_index(drop=True)

print(f"\nTipos DEPOIS da transformação:")
print(df_fin.dtypes)

In [ ]:
df_fin.head()

## 2.3 — Transformação do Ibovespa

In [ ]:
# Padronizar coluna de data
if "Date" in df_ibov.columns:
    df_ibov = df_ibov.rename(columns={"Date": "date"})
df_ibov["date"] = pd.to_datetime(df_ibov["date"])

# Garantir tipos numéricos
numeric_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
for col in numeric_cols:
    if col in df_ibov.columns:
        df_ibov[col] = pd.to_numeric(df_ibov[col], errors="coerce")

# Remover linhas com fechamento nulo
df_ibov = df_ibov.dropna(subset=["Close"])
df_ibov = df_ibov.sort_values("date").reset_index(drop=True)

# Calcular retorno diário
df_ibov["retorno_diario"] = df_ibov["Close"].pct_change()

print(f"Ibovespa limpo: {len(df_ibov)} registros")
df_ibov.head()

## 2.4 — Transformação do Orçamento

In [ ]:
df_orc["mes"] = pd.to_datetime(df_orc["mes"])
df_orc["receita"] = df_orc["receita"].astype("float64")
df_orc["despesa"] = df_orc["despesa"].astype("float64")
df_orc["saldo"] = df_orc["saldo"].astype("float64")

print(f"Orçamento limpo: {len(df_orc)} registros")
df_orc.dtypes

## 2.5 — Detecção de Anomalias (Z-score, limiar 3σ)

Aplicamos o método estatístico do **Z-score** sobre a coluna `amount` do dataset financeiro. Valores com |Z| > 3 são classificados como anomalias (outliers). Isso corresponde a pontos que estão a mais de 3 desvios-padrão da média.

In [ ]:
# Calcular Z-scores
z_scores = np.abs(stats.zscore(df_fin["amount"]))
df_fin["z_score"] = z_scores
df_fin["anomalia"] = z_scores > 3  # limiar 3σ

n_anomalias = df_fin["anomalia"].sum()
pct_anomalias = (n_anomalias / len(df_fin)) * 100

print(f"Anomalias detectadas: {n_anomalias} ({pct_anomalias:.1f}% do total)")
print(f"Média dos valores:    R$ {df_fin['amount'].mean():,.2f}")
print(f"Desvio-padrão:        R$ {df_fin['amount'].std():,.2f}")
print(f"Limiar 3σ:            R$ {df_fin['amount'].mean() + 3 * df_fin['amount'].std():,.2f}")

In [ ]:
# Visualizar anomalias
print("\n--- Registros classificados como ANOMALIA ---")
df_fin[df_fin["anomalia"]].sort_values("amount", ascending=False)

In [ ]:
# Gráfico: distribuição com anomalias destacadas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df_fin["amount"], bins=60, color="#3b82f6", edgecolor="white", alpha=0.7)
axes[0].axvline(df_fin["amount"].mean() + 3 * df_fin["amount"].std(),
                color="red", linestyle="--", label="Limiar +3σ")
axes[0].axvline(df_fin["amount"].mean() - 3 * df_fin["amount"].std(),
                color="red", linestyle="--", label="Limiar −3σ")
axes[0].set_title("Distribuição de Valores (com limiares 3σ)")
axes[0].set_xlabel("Valor (R$)")
axes[0].set_ylabel("Frequência")
axes[0].legend()

# Scatter temporal
normal = df_fin[~df_fin["anomalia"]]
anomalo = df_fin[df_fin["anomalia"]]
axes[1].scatter(normal["date"], normal["amount"], s=6, alpha=0.3, color="#3b82f6", label="Normal")
axes[1].scatter(anomalo["date"], anomalo["amount"], s=40, color="red", zorder=5, label="Anomalia")
axes[1].set_title("Transações ao longo do tempo (anomalias em vermelho)")
axes[1].set_xlabel("Data")
axes[1].set_ylabel("Valor (R$)")
axes[1].legend()

plt.tight_layout()
plt.savefig("outputs/anomalias_zscore.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Gráfico salvo em outputs/anomalias_zscore.png")

## 2.6 — Salvar dados processados

In [ ]:
df_fin.to_parquet("data/processed/finance_clean.parquet", index=False)
df_ibov.to_parquet("data/processed/ibovespa_clean.parquet", index=False)
df_orc.to_parquet("data/processed/orcamento_clean.parquet", index=False)

print("✔ Todos os datasets processados salvos em data/processed/")
print(f"  finance_clean.parquet:  {len(df_fin)} registros ({n_anomalias} anomalias marcadas)")
print(f"  ibovespa_clean.parquet: {len(df_ibov)} registros")
print(f"  orcamento_clean.parquet: {len(df_orc)} registros")

---
## Resumo da Transformação

| Etapa | Detalhe |
|-------|--------|
| Padronização | Datas → `datetime64`, valores → `float64` |
| Limpeza | Duplicatas removidas, nulos tratados |
| Anomalias | Z-score com limiar 3σ — outliers marcados em coluna `anomalia` |
| Enriquecimento | Retorno diário calculado no Ibovespa |

**Próximo:** `03_predict_regression.ipynb` — Regressão linear para projeção de receita.